In [ ]:

# ============================================================
# 4. DATA CLEANING AND TARGET PREPARATION
# ============================================================

data = df.copy()

# Standardize column names if needed
data.columns = [c.strip() for c in data.columns]

# Remove duplicate rows
before_duplicates = data.shape[0]
data = data.drop_duplicates()
after_duplicates = data.shape[0]

print("Rows before duplicate removal:", before_duplicates)
print("Rows after duplicate removal:", after_duplicates)
print("Duplicates removed:", before_duplicates - after_duplicates)

# Keep only Employed and Unemployed for binary classification
data = data[data["Employment_Status"].isin(["Employed", "Unemployed"])].copy()

# Create binary target: Unemployed = 1, Employed = 0
data["Unemployed_Target"] = data["Employment_Status"].map({
    "Employed": 0,
    "Unemployed": 1
})

# Fill missing values
# Job_Sector has many missing values, so we keep them as a meaningful category.
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].fillna("Unknown")

for col in data.select_dtypes(include=["int64", "float64"]).columns:
    data[col] = data[col].fillna(data[col].median())

print("Final dataset after target cleaning:", data.shape)
print("\nTarget distribution:")
display(data["Unemployed_Target"].value_counts())
display(data["Employment_Status"].value_counts())


Rows before duplicate removal: 300000
Rows after duplicate removal: 299997
Duplicates removed: 3
Final dataset after target cleaning: (225382, 16)

Target distribution:


,count
Unemployed_Target,
0,156644
1,68738


,count
Employment_Status,
Employed,156644
Unemployed,68738


In [ ]:

# ============================================================
# 5. FEATURE SELECTION
# ============================================================

target_col = "Unemployed_Target"

# IMPORTANT:
# We do NOT use Job_Sector because it creates data leakage.
# In this dataset, unemployed records often have missing/Unknown Job_Sector,
# so the model can learn the answer directly and produce impossible 100% scores.
# We also do NOT use Salary because it is known after employment, not before prediction.
features = [

    "Education_Level",
    "Field_of_Study",
    "Language_Proficiency",
    "Gender",
    "Region_of_Study",
    "Age",
    "Years_Since_Graduation",
    "GPA",
    "Internship_Experience"
]

# Keep only columns that exist in the dataset
features = [col for col in features if col in data.columns]

# Extra safety: remove leakage columns if they appear accidentally
leakage_columns = ["Employment_Status", "Unemployed_Target", "Salary", "Job_Sector"]
features = [col for col in features if col not in leakage_columns]

X = data[features].copy()
y = data[target_col].copy()

categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Selected features:", features)
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print("Target:", target_col)
print("X shape:", X.shape)
print("y shape:", y.shape)


Selected features: ['Education_Level', 'Field_of_Study', 'Language_Proficiency', 'Gender', 'Region_of_Study', 'Age', 'Years_Since_Graduation', 'GPA', 'Internship_Experience']
Categorical features: ['Education_Level', 'Field_of_Study', 'Language_Proficiency', 'Gender', 'Region_of_Study', 'Internship_Experience']
Numeric features: ['Age', 'Years_Since_Graduation', 'GPA']
Target: Unemployed_Target
X shape: (225382, 9)
y shape: (225382,)


In [ ]:

# ============================================================
# 6. PREPROCESSING PIPELINE
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")


Preprocessing pipeline created successfully.


In [ ]:

# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True))
print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True))


Training records: 180305
Testing records: 45077

Training target distribution:


,proportion
Unemployed_Target,
0,0.695017
1,0.304983



Testing target distribution:


,proportion
Unemployed_Target,
0,0.695011
1,0.304989
